# Ensemble: RNN + BiLSTM (No ESM)

In [ ]:
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from tqdm import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 1. Data

In [ ]:
seq_df = pd.read_csv('data/2018-06-06-pdb-intersect-pisces.csv', usecols=['pdb_id','seq'])
lab_df = pd.read_csv('data/2018-06-06-ss.cleaned.csv', usecols=['pdb_id','sst8','sst3'])
df = pd.merge(seq_df, lab_df, on='pdb_id', how='inner')
df['seq'] = df['seq'].str.replace('*','X')
df = df.dropna(subset=['seq','sst8','sst3']).copy()
df = df[(df['seq'].str.len()==df['sst8'].str.len()) & (df['seq'].str.len()==df['sst3'].str.len())].reset_index(drop=True)
df['len'] = df['seq'].str.len()
ss8_vocab = {'H':0,'G':1,'I':2,'E':3,'B':4,'T':5,'S':6,'C':7}
ss3_vocab = {'H':0,'E':1,'C':2}
chars = sorted(set(''.join(df['seq'])))
seq_vocab = {c:i+1 for i,c in enumerate(chars)}
seq_vocab['<pad>'] = 0; vocab_size=len(seq_vocab)
vocab_size


## 2. Dataset + Loaders

In [ ]:
class DS(Dataset):
    def __init__(self, seqs, s8, s3): self.seqs,self.s8,self.s3=seqs,s8,s3
    def __len__(self): return len(self.seqs)
    def __getitem__(self,i):
        t=[seq_vocab.get(c,0) for c in self.seqs[i]]
        a=[ss8_vocab.get(c,-1) for c in self.s8[i]]
        b=[ss3_vocab.get(c,-1) for c in self.s3[i]]
        return torch.tensor(t), torch.tensor(a), torch.tensor(b)
def collate(b):
    x,a,b = zip(*b)
    return (pad_sequence(x,batch_first=True,padding_value=seq_vocab['<pad>']),
            pad_sequence(a,batch_first=True,padding_value=-1),
            pad_sequence(b,batch_first=True,padding_value=-1))
idx_tr, idx_tmp = train_test_split(range(len(df)), test_size=0.2, random_state=42)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.5, random_state=42)
ds_tr = DS(df.iloc[idx_tr]['seq'].tolist(), df.iloc[idx_tr]['sst8'].tolist(), df.iloc[idx_tr]['sst3'].tolist())
ds_va = DS(df.iloc[idx_va]['seq'].tolist(), df.iloc[idx_va]['sst8'].tolist(), df.iloc[idx_va]['sst3'].tolist())
ds_te = DS(df.iloc[idx_te]['seq'].tolist(), df.iloc[idx_te]['sst8'].tolist(), df.iloc[idx_te]['sst3'].tolist())
opt = dict(num_workers=2, pin_memory=True)
train_loader = DataLoader(ds_tr, batch_size=16, shuffle=True, collate_fn=collate, **opt)
val_loader   = DataLoader(ds_va, batch_size=16, shuffle=False, collate_fn=collate, **opt)
test_loader  = DataLoader(ds_te, batch_size=16, shuffle=False, collate_fn=collate, **opt)
embedding_dim=128; embedding_dim


## 3. Models

In [ ]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=256, layers=2, dropout=0.3):
        super().__init__(); self.pad=seq_vocab['<pad>']
        self.emb=nn.Embedding(vocab_size, embed_dim, padding_idx=self.pad)
        self.rnn=nn.RNN(embed_dim, hidden, num_layers=layers, nonlinearity='tanh', bidirectional=False, batch_first=True, dropout=dropout if layers>1 else 0.0)
        self.drop=nn.Dropout(dropout); self.q8=nn.Linear(hidden,8); self.q3=nn.Linear(hidden,3)
    def forward(self,x):
        e=self.emb(x); L=(x!=self.pad).sum(1).to(torch.int64).cpu(); orig=e.size(1)
        p=pack_padded_sequence(e,L,batch_first=True,enforce_sorted=False)
        o,_=self.rnn(p); x,_=pad_packed_sequence(o,batch_first=True,total_length=orig)
        x=self.drop(x); return self.q8(x), self.q3(x)
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden=256, dropout=0.3):
        super().__init__(); self.pad=seq_vocab['<pad>']
        self.emb=nn.Embedding(vocab_size, embed_dim, padding_idx=self.pad)
        self.l1=nn.LSTM(embed_dim, hidden, bidirectional=True, batch_first=True)
        self.l2=nn.LSTM(hidden*2, hidden, bidirectional=True, batch_first=True)
        self.drop=nn.Dropout(dropout); self.q8=nn.Linear(hidden*2,8); self.q3=nn.Linear(hidden*2,3)
    def forward(self,x):
        e=self.emb(x); L=(x!=self.pad).sum(1).to(torch.int64).cpu(); orig=e.size(1)
        p,_=self.l1(pack_padded_sequence(e,L,batch_first=True,enforce_sorted=False)); p,_=self.l2(p); x,_=pad_packed_sequence(p,batch_first=True,total_length=orig)
        x=self.drop(x); return self.q8(x), self.q3(x)
rnn = RNN(vocab_size=vocab_size, embed_dim=embedding_dim).to(device)
bilstm = BiLSTM(vocab_size=vocab_size, embed_dim=embedding_dim).to(device)
if torch.cuda.device_count()>1: rnn=nn.DataParallel(rnn); bilstm=nn.DataParallel(bilstm)
rnn, bilstm


## 4. Train

In [ ]:
def acc(logits, y): p=logits.argmax(-1); m=y!=-1; return (p[m]==y[m]).float().mean().item() if m.any() else 0.0
q3_id_to_char={0:'H',1:'E',2:'C'}
def get_segments(chars,state): seg=[]; st=-1
    
    for i,ch in enumerate(chars):
        if ch==state:
            if st==-1: st=i
        elif st!=-1: seg.append((st,i-1)); st=-1
    if st!=-1: seg.append((st,len(chars)-1)); return seg
def sov_q3(logits,y):
    p=logits.argmax(-1); B=p.size(0); s=0.0
    for i in range(B):
        m=y[i]!=-1; pc=[q3_id_to_char.get(t.item(),'C') for t in p[i][m]]; tc=[q3_id_to_char.get(t.item(),'C') for t in y[i][m]]
        tw=0.0; tr=0.0
        for st in ['H','E','C']:
            ts=get_segments(tc,st); ps=get_segments(pc,st); n=sum(ch==st for ch in tc); tr+=n
            if not ts: continue
            for a,b in ts:
                lo=b-a+1; best=0; bestM=lo; bestL=0
                for c,d in ps:
                    o=max(0,min(b,d)-max(a,c)+1)
                    if o>0:
                        M=max(b,d)-min(a,c)+1; L=d-c+1
                        if o>best: best=o; bestM=M; bestL=L
                if best>0:
                    delta=min(bestM-best,best,lo//2,bestL//2); tw+= (best+delta)/bestM * lo
        if tr>0: s+=tw/tr
    return s/max(1,B)
def train_one(model, name, ckpt):
    c8=nn.CrossEntropyLoss(ignore_index=-1); c3=nn.CrossEntropyLoss(ignore_index=-1); opt=torch.optim.Adam(model.parameters(), lr=1e-4)
    best=-1.0
    for ep in range(1,21):
        model.train(); tl=ta8=ta3=tsv=0.0
        for x,s8,s3 in tqdm(train_loader, desc=f'{name} Epoch {ep}/20', leave=False):
            x,s8,s3=x.to(device),s8.to(device),s3.to(device)
            q8,q3=model(x)
            l8=c8(q8.view(-1,8), s8.view(-1)); l3=c3(q3.view(-1,3), s3.view(-1)); loss=l8+0.5*l3
            opt.zero_grad(); loss.backward(); opt.step()
            tl+=loss.item(); ta8+=acc(q8,s8); ta3+=acc(q3,s3); tsv+=sov_q3(q3,s3)
        tl/=len(train_loader); ta8/=len(train_loader); ta3/=len(train_loader); tsv/=len(train_loader)
        model.eval(); vl=va8=va3=vsv=0.0
        with torch.no_grad():
            for x,s8,s3 in val_loader:
                x,s8,s3=x.to(device),s8.to(device),s3.to(device)
                q8,q3=model(x); l8=c8(q8.view(-1,8), s8.view(-1)); l3=c3(q3.view(-1,3), s3.view(-1)); loss=l8+0.5*l3
                vl+=loss.item(); va8+=acc(q8,s8); va3+=acc(q3,s3); vsv+=sov_q3(q3,s3)
        vl/=len(val_loader); va8/=len(val_loader); va3/=len(val_loader); vsv/=len(val_loader)
        print(f'{name} Epoch {ep}: Train Loss={tl:.4f}, Val Loss={vl:.4f}')
        print(f'Train Acc Q8={ta8:.4f}, Val Acc Q8={va8:.4f}')
        print(f'Train Acc Q3={ta3:.4f}, Val Acc Q3={va3:.4f}, Train SOV Q3={tsv:.4f}, Val SOV Q3={vsv:.4f}')
        if va8>best: best=va8; torch.save(model.module.state_dict() if isinstance(model,nn.DataParallel) else model.state_dict(), ckpt)
    model.load_state_dict(torch.load(ckpt, map_location='cpu')); model.to(device); model.eval()
train_one(rnn,'RNN','best_rnn_noemb.pt')
train_one(bilstm,'BiLSTM','best_bilstm_noemb.pt')
'trained'


## 5. Ensemble

In [ ]:
@torch.no_grad()
def collect(loader):
    q8r,q3r,q8b,q3b,s8s,s3s=[],[],[],[],[],[]
    for x,s8,s3 in loader:
        x,s8,s3=x.to(device),s8.to(device),s3.to(device)
        a8,a3=rnn(x); b8,b3=bilstm(x)
        q8r.append(a8.cpu()); q3r.append(a3.cpu()); q8b.append(b8.cpu()); q3b.append(b3.cpu()); s8s.append(s8.cpu()); s3s.append(s3.cpu())
    import torch as T
    return T.cat(q8r),T.cat(q3r),T.cat(q8b),T.cat(q3b),T.cat(s8s),T.cat(s3s)
def masked_acc(logits, labels):
    p=logits.argmax(-1); m=labels!=-1; import torch as T;
    return (p[m]==labels[m]).float().mean().item() if m.any() else 0.0
v_q8r,v_q3r,v_q8b,v_q3b,v_s8,v_s3=collect(val_loader)
import torch as T
alphas=T.linspace(0,1,21); best_a8=0.5; best_v8=-1; best_a3=0.5; best_v3=-1
for a in alphas:
    acc8=masked_acc(a*v_q8r+(1-a)*v_q8b, v_s8); acc3=masked_acc(a*v_q3r+(1-a)*v_q3b, v_s3)
    if acc8>best_v8: best_v8, best_a8 = acc8, float(a)
    if acc3>best_v3: best_v3, best_a3 = acc3, float(a)
print(f'Best alpha Q8: {best_a8:.2f} | Val Acc: {best_v8:.4f}')
print(f'Best alpha Q3: {best_a3:.2f} | Val Acc: {best_v3:.4f}')
@torch.no_grad()
def evaluate(loader,a8,a3):
    tot8=tot3=0; cor8=cor3=0
    for x,s8,s3 in loader:
        x,s8,s3=x.to(device),s8.to(device),s3.to(device)
        r8,r3=rnn(x); b8,b3=bilstm(x)
        q8=a8*r8+(1-a8)*b8; q3=a3*r3+(1-a3)*b3
        p8=q8.argmax(-1); p3=q3.argmax(-1); m8=s8!=-1; m3=s3!=-1
        cor8+=(p8[m8]==s8[m8]).sum().item(); tot8+=m8.sum().item(); cor3+=(p3[m3]==s3[m3]).sum().item(); tot3+=m3.sum().item()
    return cor8/max(1,tot8), cor3/max(1,tot3)
t8,t3=evaluate(test_loader,best_a8,best_a3)
print(f'Test Accuracy Q8: {t8:.4f}')
print(f'Test Accuracy Q3: {t3:.4f}')
